In [ ]:
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import LocallyLinearEmbedding, MDS
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score

#import PC1, PC2, PC3 
def load_arff_dataset(path):
    data, meta = arff.loadarff(path)
    df = pd.DataFrame(data)


    # Last column is target
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    # Encode labels
    if y.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X.values, y
#reduce_dim(method, X_train, x_test, n_compents) call for PCA, Kernal PCA, LLE, MDS and 20,15,10,5
def reduce_dim(method, X_train, X_test, n_components):
    if method == "pca":
        model = PCA(n_components=n_components)

    elif method == "kpca":
        model = KernelPCA(n_components=n_components, kernel="rbf")

    elif method == "lle":
        model = LocallyLinearEmbedding(
            n_components=n_components,
            n_neighbors=10,
            eigen_solver="dense"
        )
  #  elif method == "mds":
   #     model = MDS(n_components=n_components)

        X_train_red = model.fit_transform(X_train)
        X_test_red = model.fit_transform(X_test)
        return X_train_red, X_test_red
        
    X_train_red = model.fit_transform(X_train)
    X_test_red = model.transform(X_test)

    return X_train_red, X_test_red

#run classifaction model on test data for logistic regression, random Forest, and SVM
def evaluate_models(X_train, X_test, y_train, y_test):
    models = {
        "logreg": LogisticRegression(max_iter=1000),
        "rf": RandomForestClassifier(),
        "svm": SVC()
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

        results.append({
            "model": name,
            "precision": precision,
            "recall": recall
        })

    return results
    
#create pipeline
def run_pipeline(dataset_path):
    X, y = load_arff_dataset(dataset_path)

    # Split 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    methods = ["pca", "kpca", "lle"]
    components_list = [20, 15, 10, 5]

    all_results = []

    for method in methods:
        for n in components_list:
            
                X_train_red, X_test_red = reduce_dim(method, X_train, X_test, n)

                model_results = evaluate_models(
                    X_train_red, X_test_red, y_train, y_test
                )

                for res in model_results:
                    res.update({
                        "dataset": dataset_path,
                        "dim_method": method,
                        "n_components": n
                    })
                    all_results.append(res)

    return pd.DataFrame(all_results)

#run pipeline on PC1, PC2, PC3 to preform dim reduction, classification, calc precision, and clac recall
def run_all_datasets():
    datasets = [
        "php8g7WIL.arff",
        "phplybWZX.arff",
        "phppQG1iG.arff"
    ]

    final_results = pd.DataFrame()

    for ds in datasets:
        df = run_pipeline(ds)
        final_results = pd.concat([final_results, df], ignore_index=True)

    return final_results

#evalulate which is the best config for each data set
def find_best_configs(results_df):
    best_configs = results_df.sort_values(
        by=["precision", "recall"], ascending=False
    ).groupby("dataset").first()

    return best_configs

if __name__ == "__main__":
    results = run_all_datasets()

    print("\nAll Results:")
    print(results)

    best = find_best_configs(results)

    print("\nBest Configurations per Dataset:")
    print(best)
